In [ ]:
import csv
import logging
import os
import json
import random
import re
import shutil
import subprocess
import time
import threading
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Set, Dict

import undetected_chromedriver as uc
from dotenv import load_dotenv
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

load_dotenv()

BASE_DIR = Path("/home/kongla/Documents/GitHub/Real-estate-Scraping")
OUTPUT_PATH = BASE_DIR / "facebook" / "facebook_output.csv"
PROFILE_PATH = BASE_DIR / "chrome_profile"

TARGET_POSTS = 120
MAX_STAGNANT = 10
SCROLL_SIZE = 3000

MODEL_NAME = "typhoon-v2.5-30b-a3b-instruct"
MAX_WORKERS = 5
LLM_TIMEOUT = 60.0

client = OpenAI(api_key=os.getenv("TYPHOON_API_KEY"), base_url="https://api.opentyphoon.ai/v1", timeout=LLM_TIMEOUT)

GROUP_URLS: List[str] = [
    "https://www.facebook.com/share/g/17yvNkWcJG/",
    "https://www.facebook.com/share/g/17cuAKX5Py/",
    "https://www.facebook.com/share/g/1Wm4GxnHFm/",
    "https://www.facebook.com/share/g/1Dp8rpYxmE/",
    "https://www.facebook.com/share/g/1F3cnPDmGU/",
    "https://www.facebook.com/share/g/1N2eXd6ge5/",
    "https://www.facebook.com/share/g/1AnUsnRNod/",
    "https://www.facebook.com/share/g/1JyWvsFtu6/",
    "https://www.facebook.com/share/g/1AzBzdPhEL/",
    "https://www.facebook.com/share/g/18NsbCsA52/",
    "https://www.facebook.com/share/g/1CKirSUbzs/",
    "https://www.facebook.com/share/g/173xAigqGb/",
    "https://www.facebook.com/share/g/17yZmR3H1U/",
    "https://www.facebook.com/share/g/1EV9v3JxPX/",
    "https://www.facebook.com/share/g/1BXjpfQN97/",
    "https://www.facebook.com/share/g/18Mh4scEdr/",
    "https://www.facebook.com/share/g/17rf5Ppt8a/",
    "https://www.facebook.com/share/g/1EGJYxCSeT/",
    "https://www.facebook.com/share/g/1C9S5cRLgw/",
    "https://www.facebook.com/share/g/1FgHtiV8oF/",
    "https://www.facebook.com/groups/485019775925280/",
    "https://www.facebook.com/share/g/1AgeQ1WpzA/",
    "https://www.facebook.com/share/g/1XYiLFYJgX/",
    "https://www.facebook.com/share/g/1CAAbCJJVT/",
    "https://www.facebook.com/share/g/1879ZUrbpH/",
    "https://www.facebook.com/share/g/1Bhed5kut5/",
    "https://www.facebook.com/share/g/1J75CA6Aox/",
    "https://www.facebook.com/share/g/1CVwGydoVb/",
    "https://www.facebook.com/share/g/1SKVJR95jx/",
    "https://www.facebook.com/share/g/1Di1RbGM5Z/",
    "https://www.facebook.com/share/g/1AgFYb1qH7/",
    "https://www.facebook.com/share/g/1WhmM13MbK/",
    "https://www.facebook.com/share/g/1Ch6CSbFLq/",
    "https://www.facebook.com/share/g/19vVjxviuP/",
    "https://www.facebook.com/share/g/189YXg4LZr/",
    "https://www.facebook.com/share/g/1HmmyqcGTq/",
    "https://www.facebook.com/share/g/18E2JPkYMH/",
    "https://www.facebook.com/share/g/1ArRmRHKEy/",
    "https://www.facebook.com/share/g/19c9HUsnBN/",
    "https://www.facebook.com/share/g/1GTM17dYew/",
    "https://www.facebook.com/share/g/189gkv2NVu/",
    "https://www.facebook.com/share/g/1BGgqYXFrJ/",
    "https://www.facebook.com/share/g/1E2rpaskGy/",
    "https://www.facebook.com/share/g/17iWwCY6Pc/",
    "https://www.facebook.com/share/g/1biieSTrft/",
    "https://www.facebook.com/share/g/1DmTmzs5Dn/",
    "https://www.facebook.com/share/g/1GDtgBoYjd/",
    "https://www.facebook.com/share/g/17o4QPYZfr/",
    "https://www.facebook.com/share/g/1AwtrUdMkw/",
    "https://www.facebook.com/share/g/1DFcTDtguE/",
    "https://www.facebook.com/share/g/14cTZ1dxvTo/",
    "https://www.facebook.com/share/g/1ENBMKVPrw/",
    "https://www.facebook.com/share/g/1Dksryieym/",
    "https://www.facebook.com/share/g/1DAyY5idaF/",
    "https://www.facebook.com/share/g/1CNm4wbqSe/",
    "https://www.facebook.com/share/g/17uw8vv7vb/",
    "https://www.facebook.com/share/g/1GPMEpyZEa/",
    "https://www.facebook.com/share/g/1DEHfe3cS3/",
    "https://www.facebook.com/share/g/1GZhnbeEVJ/",
    "https://www.facebook.com/share/g/1CUhdjzY2y/",
    "https://www.facebook.com/share/g/17G11FQ4MS/",
    "https://www.facebook.com/share/g/1Kmt1wQmrZ/",
    "https://www.facebook.com/share/g/1MhTuADusV/",
    "https://www.facebook.com/share/g/1CTofFNtcq/",
    "https://www.facebook.com/share/g/18A2nRTGhd/",
    "https://www.facebook.com/share/g/1AsSXshu4s/",
    "https://www.facebook.com/groups/175356699802854/",
    "https://www.facebook.com/share/g/18JNQPA5PU/",
    "https://www.facebook.com/share/g/18NPYCEess/",
    "https://www.facebook.com/share/g/1EKebYgt5Q/",
    "https://www.facebook.com/share/g/185ytLwwFg/",
    "https://www.facebook.com/share/g/1Dv3yJSrK4/",
    "https://www.facebook.com/share/g/17G6mCkmKJ/",
    "https://www.facebook.com/share/g/1FkNPpjiQT/",
    "https://www.facebook.com/share/g/1CK38hkJKi/",
    "https://www.facebook.com/share/g/1GTKcMsvPH/",
    "https://www.facebook.com/share/g/16kde1JhfD/",
    "https://www.facebook.com/share/g/1GdLnRnNa8/",
    "https://www.facebook.com/share/g/14VETLLJxFk/",
    "https://www.facebook.com/share/g/183AURyK31/",
    "https://www.facebook.com/share/g/1ATGortt8h/",
    "https://www.facebook.com/share/g/1briWbWRkE/",
    "https://www.facebook.com/share/g/18PynzrLZs/",
    "https://www.facebook.com/share/g/1Am4xiU2gK/",
    "https://www.facebook.com/share/g/17DJKPBtHd/",
    "https://www.facebook.com/groups/propertydit/",
    "https://www.facebook.com/share/g/185RyKmRPE/",
    "https://www.facebook.com/share/g/1CY3Mbb748/",
    "https://www.facebook.com/share/g/1B8oxbjTvm/",
    "https://www.facebook.com/share/g/1GSXJfPHRv/",
    "https://www.facebook.com/share/g/1M1HGCPGcu/",
    "https://www.facebook.com/share/g/1TNPq3ht9y/",
    "https://www.facebook.com/groups/2234744070053568/",
    "https://www.facebook.com/share/g/1CQGfxHye9/",
    "https://www.facebook.com/share/g/1DSPbvtBao/",
    "https://www.facebook.com/share/g/1GSvzLRBLr/",
    "https://www.facebook.com/share/g/1F8jgQaN2A/",
    "https://www.facebook.com/share/g/1HbNgxFjuL/",
    "https://www.facebook.com/share/g/1C88zkj62s/",
    "https://www.facebook.com/share/g/1Av5UuLqoH/",
    "https://www.facebook.com/share/g/1SmN6nuyk2/",
    "https://www.facebook.com/share/g/17H6RfoLRU/",
    "https://www.facebook.com/share/g/1cLKmAvPxn/",
    "https://www.facebook.com/share/g/174RKdW4tb/",
    "https://www.facebook.com/share/g/19WUdWcuDX/",
    "https://www.facebook.com/share/g/1Y2vcMpTCi/",
    "https://www.facebook.com/share/g/1CE4tovrbk/",
    "https://www.facebook.com/groups/1756597841090354/",
    "https://www.facebook.com/groups/theoneestate/",
    "https://www.facebook.com/groups/1512942549024788/",
    "https://www.facebook.com/share/g/1N8fiVtKft/",
    "https://www.facebook.com/share/g/17148ebYu2/",
    "https://www.facebook.com/share/g/1MeEAycCK9/",
    "https://www.facebook.com/share/g/1SZj2CdzZG/",
    "https://www.facebook.com/share/g/1J6mpUjhH1/",
    "https://www.facebook.com/share/g/1FBhAHeU2C/",
    "https://www.facebook.com/share/g/18PtvBe5Rk/",
    "https://www.facebook.com/share/g/1CV291GDMX/",
    "https://www.facebook.com/share/g/1ExkvVre1Z/",
    "https://www.facebook.com/groups/210390739740584/",
    "https://www.facebook.com/share/g/1E4S1igybj/",
    "https://www.facebook.com/share/g/1CKJ9gZFiE/",
    "https://www.facebook.com/share/g/1EMvFAXfAG/",
    "https://www.facebook.com/share/g/1BT3jt6Gzc/",
    "https://www.facebook.com/share/g/1AbuUgw43X/",
    "https://www.facebook.com/share/g/1GZJbNyZ7W/",
    "https://www.facebook.com/share/g/18GwgqjiBc/",
    "https://www.facebook.com/share/g/1HqpNUtH8B/",
    "https://www.facebook.com/share/g/1GXwMupTDL/",
    "https://www.facebook.com/share/g/1JTxjApoKP/",
    "https://www.facebook.com/share/g/19VteKNLoD/",
    "https://www.facebook.com/share/g/1C6x2GoH7u/",
    "https://www.facebook.com/share/g/1DaoFJMJkP/",
    "https://www.facebook.com/share/g/1DGSJAvihX/",
    "https://www.facebook.com/share/g/1AjZE6ZfSd/",
    "https://www.facebook.com/share/g/1DQok3tTUH/",
    "https://www.facebook.com/share/g/1C6oYUM8aD/",
    "https://www.facebook.com/share/g/1GEPncCTh9/",
    "https://www.facebook.com/share/g/1DjbxnhQBA/",
    "https://www.facebook.com/share/g/18Jr24GQVM/",
    "https://www.facebook.com/share/g/186TqCt9nn/",
    "https://www.facebook.com/share/g/1LKtYvTQyb/",
    "https://www.facebook.com/share/g/1HcXDi1ghz/",
    "https://www.facebook.com/share/g/1C1PfcEEU9/",
    "https://www.facebook.com/share/g/18JoUsmbRP/",
    "https://www.facebook.com/share/g/16wHaXjF3W/",
    "https://www.facebook.com/share/g/17AZ9awNPP/",
    "https://www.facebook.com/groups/770357389784713/",
    "https://www.facebook.com/share/g/1CB7iUsP7j/",
    "https://www.facebook.com/groups/condomarket/",
    "https://www.facebook.com/share/g/18EVu9X5hn/",
    "https://www.facebook.com/share/g/1KJZowXjYW/",
    "https://www.facebook.com/share/g/1Akg539Dgx/",
    "https://www.facebook.com/share/g/1L5WbUZqNR/",
    "https://www.facebook.com/groups/914277069917114/",
    "https://www.facebook.com/share/g/18MYU47ZQy/",
    "https://www.facebook.com/groups/1516977068738748/",
    "https://www.facebook.com/share/g/18akkmMoGG/",
]

OUTPUT_HEADERS = [
    "วันที่โพส", "website", "ประเภท", "สถานะ", "ชื่อโครงการ",
    "ขนาด", "ราคา", "เขต", "Link", "เบอร์โทรศัพท์", "Line", "คำอธิบาย"
]

SYSTEM_PROMPT = """คุณคือ AI วิเคราะห์อสังหาริมทรัพย์ระดับ Expert รองรับการวิเคราะห์ได้ทั้ง English, Thai และ Chinese
หน้าที่ของคุณคือดึงข้อมูล (Data Extraction) และจำแนกประเภทผู้โพสต์ (Classification) จากข้อความที่ให้มา
ตอบกลับเป็น JSON Structure เท่านั้น ห้ามมี Text อื่นปน

{
  "is_real_estate": true/false,
  "is_owner": true/false,
  "owner_confidence": 0.0,
  "evidence_phrases": [],
  "risk_flags": [],
  "post_date_text": "ดึงข้อความเวลาที่พบในข้อมูลตามที่ส่งมา",
  "extracted":{
    "property_type":"",
    "rental_sale_status":"",
    "project_name":"",
    "district":"",
    "size_text":"",
    "price_text":"",
    "price_value_thb":null,
    "phone":"",
    "line":"",
    "description":"ดึงรายละเอียดข้อความทั้งหมดมา ห้ามตัดทิ้ง ห้าม Truncate เด็ดขาด"
  }
}

=== OWNER vs AGENT OPTIMIZED CLASSIFICATION (FAST-FAIL PIPELINE) ===

กลไกการตัดสิน (Short-Circuit Evaluation):
ประเมินตามลำดับ GATE 1 -> GATE 2 -> GATE 3
*** กฎเหล็ก: หากตรงกับ GATE 1 (Agent) ให้ตั้งค่า `is_owner: false`, `owner_confidence: 0.0`, และระบุ `risk_flags` ให้ชัดเจน แต่ "บังคับดึงข้อมูลใน Object `extracted` มาทั้งหมดห้ามทิ้งเด็ดขาด" เผื่อใช้ตรวจสอบย้อนหลังใน Log ***

GATE 1: AGENT HARD-FILTER (High Risk -> ถือเป็น Agent)
หากพบเงื่อนไขใดเงื่อนไขหนึ่งต่อไปนี้ ถือเป็น Agent/Sales แน่นอน (is_owner: false):
- [Line Official Account]: การระบุ LINE ID ที่มีเครื่องหมาย "@" นำหน้า เช่น "LINE : @homecareproperty", "@Dgrandhouse" ถือเป็น Agent 100%
- [Pattern/Repeated Contact]: มีการระบุรหัสทรัพย์สิน (Property ID), รูปแบบการโพสต์เป็น Template ซ้ำๆ, หรือใช้ Hashtag จำนวนมากผิดปกติ
- [Sales Closing]: "ปิดเกม", "Units สุดท้าย", "โค้งสุดท้าย", "จองด่วน", "Hot Item", "Rare Item", "เปิดรับลงทะเบียน", "รอบ VVIP"
- [Broker Services]: "บริการด้านอสังหา", "ดันสินเชื่อทุกเคส", "ฟรีค่าโอน", "ดูแลจนถึงวันโอน", "รับฝากขาย/เช่า", "Co-broker ยินดี" (ยกเว้นเจ้าของบอกรับเอเจ้นท์ ให้ดู Gate 2)
- [Corporate Marketing]: "มรดกแห่งชีวิต", "นิยามใหม่แห่งการพักผ่อน", "ยกระดับการใช้ชีวิต", "Ultra Luxury" 
- [Agent Naming/Routing]: "ทักหา[ชื่อ]", "ติดต่อคุณ...", "แอดไลน์หาทีมงาน", "แอดมิน"

GATE 2: OWNER VERIFIED (High Confidence -> เจ้าของ 100%)
หากผ่าน Gate 1 มาได้ และพบสัญญาณเหล่านี้ (is_owner: true, Confidence 0.9-1.0):
- [Direct Claim]: "เจ้าของขายเอง", "เจ้าของปล่อยเอง", "Owner Post", "เจ้าของ ยินดีรับเอเจ้นท์"
- [Ownership Proof]: "บ้านสร้างเอง", "เจ้าของไม่เคยเข้าอยู่", "ขายเพราะย้ายงาน", "อยู่เองสะอาดมาก", "ซื้อไว้นานไม่ได้อยู่", "ลดกว่าที่ซื้อมา"
- [Personal Tone]: ใช้สรรพนาม "ผม/ฉัน/พี่", "ตัดใจปล่อยด่วน", "ของจริงสวยกว่ารูป", "แถมเครื่องใช้ไฟฟ้าตามภาพ", ติดต่อ Line แบบ Personal ID (ไม่มี @)

GATE 3: HEURISTIC SCORING (Ambiguous / Neutral)
หากไม่มีสัญญาณชัดเจนจาก Gate 1 และ 2 ให้วิเคราะห์จากรูปแบบ (Pattern):
- [Pattern Agent]: Emoji ตกแต่งจนเหมือน Template หรือ Format เป็นระเบียบเกินไป -> จัดเป็น Agent (Confidence 0.1-0.3)
- [Pattern Owner]: ถ้าเป็นข้อความดิบๆ ไม่มี Format ตายตัว เล่าเหมือนคนทั่วไปพิมพ์ -> จัดเป็น Owner (Confidence 0.6-0.8)
"""

MONTH_MAP = {
    "มกราคม": 1, "ม.ค.": 1, "กุมภาพันธ์": 2, "ก.พ.": 2, "มีนาคม": 3, "มี.ค.": 3,
    "เมษายน": 4, "เม.ย.": 4, "พฤษภาคม": 5, "พ.ค.": 5, "มิถุนายน": 6, "มิ.ย.": 6,
    "กรกฎาคม": 7, "ก.ค.": 7, "สิงหาคม": 8, "ส.ค.": 8, "กันยายน": 9, "ก.ย.": 9,
    "ตุลาคม": 10, "ต.ค.": 10, "พฤศจิกายน": 11, "พ.ย.": 11, "ธันวาคม": 12, "ธ.ค.": 12,
}

csv_lock = threading.Lock()

def get_chrome_version(chrome_exec: str) -> int:
    try:
        res = subprocess.run([chrome_exec, "--version"], capture_output=True, text=True, check=False)
        return int(re.search(r"(\d+)\.", res.stdout).group(1)) if res.stdout else 0
    except Exception:
        return 0

def create_driver() -> uc.Chrome:
    PROFILE_PATH.mkdir(parents=True, exist_ok=True)
    chrome_exec = shutil.which("google-chrome") or shutil.which("chromium-browser")
    opts = uc.ChromeOptions()
    opts.add_argument(f"--user-data-dir={PROFILE_PATH}")
    opts.add_argument("--disable-notifications")
    opts.page_load_strategy = "eager"
    return uc.Chrome(options=opts, version_main=get_chrome_version(chrome_exec), browser_executable_path=chrome_exec)

def humanized_scroll(driver: uc.Chrome) -> None:
    driver.execute_script(f"window.scrollBy(0, {SCROLL_SIZE + random.randint(-500, 500)});")
    time.sleep(random.uniform(1.5, 3.0))

def apply_new_post_filter(driver: uc.Chrome):
    try:
        driver.execute_script("""
            const filterBtn = Array.from(document.querySelectorAll('div[role="button"]'))
                .find(e => e.innerText && (e.innerText.includes('เรียงลำดับฟีดในกลุ่มตาม') || e.innerText.includes('จัดเรียงตาม')));
            if (filterBtn) {
                filterBtn.click();
                setTimeout(() => {
                    const options = Array.from(document.querySelectorAll('div[role="menuitemradio"]'));
                    const target = options.find(e => e.innerText && (e.innerText.includes('โพสต์ใหม่') || e.innerText.includes('รายการสินค้าใหม่')));
                    if (target) target.click();
                }, 1500);
            }
        """)
        time.sleep(3.5)
    except Exception as e:
        logger.error(f"Filter error: {e}")

def expand_all_see_more(driver: uc.Chrome):
    try:
        driver.execute_script("""
            Array.from(document.querySelectorAll('div[role="button"]')).forEach(b => {
                const txt = (b.textContent || "").trim();
                if (txt === 'ดูเพิ่มเติม' || txt === 'See more') {
                    b.click();
                }
            });
        """)
        time.sleep(1.5) 
    except Exception as e:
        pass

def batch_extract_dom(driver: uc.Chrome) -> List[Dict[str, str]]:
    return driver.execute_script("""
        const results = [];
        document.querySelectorAll("div[role='article']").forEach(a => {
            const linkNodes = Array.from(a.querySelectorAll("a[href]")).filter(l => l.href.includes('/posts/') || l.href.includes('/permalink/'));
            if (linkNodes.length === 0) return;
            
            const linkNode = linkNodes[0];
            const url = linkNode.href.split('?')[0];
            const msgNode = a.querySelector("div[data-ad-comet-preview='message']") || a.querySelector("div[data-ad-preview='message']");
            if (!msgNode) return;
            
            const content = msgNode.innerText.trim();
            let date = "N/A";
            
            for (let l of linkNodes) {
                const aria = (l.getAttribute("aria-label") || "").trim();
                const text = (l.textContent || "").trim();
                if (aria && aria.length > 0 && aria.length < 30) {
                    date = aria;
                    break;
                } else if (text && text.length > 0 && text.length < 30) {
                    date = text;
                    break;
                }
            }
            
            results.push({"Post_URL": url, "Full_Content": content, "Date": date});
        });
        return results;
    """)

def call_llm_service(payload: str) -> dict | None:
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=0.0,
                max_tokens=15000,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": payload},
                ],
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            time.sleep(2 ** attempt)
    return None

def parse_date(date_text: str) -> str:
    now = datetime.now()
    if not date_text or date_text == "N/A": return "-"
    val = date_text.strip().lower()
    
    if any(k in val for k in ("วันนี้", "นาที", "ชั่วโมง", "ชม.")):
        return now.strftime("%d/%m/%Y")
    
    if "เมื่อวาน" in val:
        return (now - timedelta(days=1)).strftime("%d/%m/%Y")
    
    m_days = re.search(r"(\d+)\s*วัน", val)
    if m_days:
        return (now - timedelta(days=int(m_days.group(1)))).strftime("%d/%m/%Y")
    
    m_date = re.search(r"(\d{1,2})[\s\.\-/]+([ก-๙a-zA-Z]+)(?:[\s\.\-/]+(\d{2,4}))?", val)
    if m_date:
        d, m_raw = int(m_date.group(1)), m_date.group(2)
        m = MONTH_MAP.get(m_raw)
        if m:
            y = now.year
            if m_date.group(3):
                y_raw = int(m_date.group(3))
                y = y_raw - 543 if y_raw > 2400 else (y_raw if y_raw > 100 else 2000 + y_raw)
            return f"{d:02d}/{m:02d}/{y}"
            
    return "-"

def transform_record(raw_row: dict, ai_data: dict) -> dict:
    ext = ai_data.get("extracted", {})
    llm_date = ai_data.get("post_date_text", "").strip()
    dom_date = raw_row.get("Date", "").strip()
    
    actual_date = llm_date if llm_date and llm_date != "N/A" else dom_date
    
    return {
        "วันที่โพส": parse_date(actual_date),
        "website": "facebook",
        "ประเภท": ext.get("property_type", "-"),
        "สถานะ": ext.get("rental_sale_status", "-"),
        "ชื่อโครงการ": ext.get("project_name", "-"),
        "ขนาด": ext.get("size_text", "-"),
        "ราคา": ext.get("price_text", "-"),
        "เขต": ext.get("district", "-"),
        "Link": raw_row.get("Post_URL", "-"),
        "เบอร์โทรศัพท์": ext.get("phone", "-"),
        "Line": ext.get("line", "-"),
        "คำอธิบาย": ext.get("description", "-"),
    }

def worker_process_and_save(raw_item: dict) -> None:
    payload = f"Post Date: {raw_item.get('Date', 'N/A')}\n\nContent:\n{raw_item.get('Full_Content', '')}"
    
    ai_response = call_llm_service(payload)
    if not ai_response or not ai_response.get("is_real_estate"):
        return
    
    if not ai_response.get("is_owner"):
        logger.info(f"Agent Filtered | URL: {raw_item.get('Post_URL')} | Pattern: {ai_response.get('risk_flags')}")
        return

    final_data = transform_record(raw_item, ai_response)
    with csv_lock:
        with open(OUTPUT_PATH, 'a', encoding='utf-8-sig', newline='') as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writerow(final_data)

def process_group(driver: uc.Chrome, url: str, seen_urls: Set[str], executor: ThreadPoolExecutor, group_idx: int, total_groups: int):
    try:
        logger.info(f"[Group {group_idx}/{total_groups}] Start processing: {url}")
        driver.get(url)
        time.sleep(5)
        apply_new_post_filter(driver)
        
        saved_count, stagnant_count = 0, 0
        for _ in range(300):
            expand_all_see_more(driver)
            
            extracted = batch_extract_dom(driver)
            if not extracted:
                stagnant_count += 1
            else:
                new_items = [i for i in extracted if i["Post_URL"] not in seen_urls]
                
                if new_items:
                    stagnant_count = 0
                    for item in new_items:
                        seen_urls.add(item["Post_URL"])
                        executor.submit(worker_process_and_save, item)
                        saved_count += 1
                    logger.info(f"[Group {group_idx}/{total_groups}] Collected {len(new_items)} new items (Total saved this group: {saved_count}/{TARGET_POSTS})")
                else:
                    stagnant_count += 1
                    
            if saved_count >= TARGET_POSTS or stagnant_count >= MAX_STAGNANT:
                logger.info(f"[Group {group_idx}/{total_groups}] Stop condition met. (Saved: {saved_count}, Stagnant: {stagnant_count})")
                break
                
            humanized_scroll(driver)
    except Exception as e:
        logger.error(f"Error processing {url}: {e}")

def main():
    if not OUTPUT_PATH.exists():
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(OUTPUT_PATH, "w", newline="", encoding="utf-8-sig") as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writeheader()

    seen_urls: Set[str] = set()
    with open(OUTPUT_PATH, "r", encoding="utf-8-sig") as f:
        seen_urls.update(row["Link"] for row in csv.DictReader(f) if row.get("Link"))

    driver = create_driver()
    try:
        total_groups = len(GROUP_URLS)
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for idx, group_url in enumerate(GROUP_URLS, 1):
                process_group(driver, group_url, seen_urls, executor, idx, total_groups)
    finally:
        driver.quit()

if __name__ == "__main__":
    main()

[01:27:34] [INFO] patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
[01:27:36] [INFO] 🚀 [Group 1/158] Start processing: https://www.facebook.com/share/g/17yvNkWcJG/
[01:27:51] [INFO] [Group 1/158] Collected 1 new items (Total saved this group: 1/120)
[01:27:55] [INFO] [Group 1/158] Collected 2 new items (Total saved this group: 3/120)
[01:28:00] [INFO] [Group 1/158] Collected 1 new items (Total saved this group: 4/120)
[01:28:08] [INFO] [Group 1/158] Collected 2 new items (Total saved this group: 6/120)
[01:28:12] [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
[01:28:16] [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
[01:28:16] [INFO] Agent Filtered | URL: https://www.facebook.com/groups/933477144626594/posts/1649124219728546/ | Pattern: ['AGENT_HARD_FILTER_NAMED_BROKER', 'AGENT_HARD_FILTER_HASHTAG_BROKER', 'AGENT_HARD_FILTER_CONTACT_METHOD_BROKER']